### Config stuff

In [13]:
import random
import pyspark
from delta.tables import IdentityGenerator
from pyspark.sql import SparkSession, functions
import ConnectionConfig as cc
from pyspark.sql.functions import *
cc.setupEnvironment()
cc.listEnvironment()

SHELL: /bin/bash
CONDA_EXE: /opt/conda/bin/conda
_CE_M: 
HOSTNAME: sparkjupyter
LANGUAGE: C.UTF-8
_START_SH_EXECUTED: 1
SPARK_OPTS: --driver-java-options=-Xms1024M --driver-java-options=-Xmx4096M --driver-java-options=-Dlog4j.logLevel=info
NB_UID: 1000
XML_CATALOG_FILES: file:///opt/conda/etc/xml/catalog file:///etc/xml/catalog
PWD: /home/jovyan
CONDA_PREFIX: /opt/conda
PYSPARK_SUBMIT_ARGS: --packages io.delta:delta-spark_2.13:4.0.0 pyspark-shell
HOME: /home/jovyan
LANG: C.UTF-8
NB_GID: 100
CONDA_PROMPT_MODIFIER: (base) 
_CONDA_EXE: /opt/conda/bin/conda
_CONDA_ROOT: /opt/conda
_CE_CONDA: 
DELTA_VERSION: 4.0.0
CONDA_SHLVL: 1
SHLVL: 0
CONDA_DIR: /opt/conda
SPARK_HOME: /usr/local/spark
CONDA_PYTHON_EXE: /opt/conda/bin/python
JUPYTER_PORT: 8888
SPARK_CONF_DIR: /usr/local/spark/conf
CONDA_DEFAULT_ENV: base
NB_USER: jovyan
LC_ALL: C.UTF-8
PATH: /opt/conda/bin:/opt/conda/condabin:/usr/local/spark/bin:/opt/conda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/local/spark/

In [14]:
spark = cc.startLocalCluster("dimSalesRepInit")
spark.getActiveSession()


# Creating the operational database
In order to run this demo the database tutorial_op has to be created following the insctructions in 04_00_CHECK_jdbcConnection.ipynb

# Initial load
We will create a slowly changing dimension type 2 called dimSalesRep based on a sourceTable in our operational database called salesrep. SCD2  tables demand extra care because we will store hirstorical values of the dimension with the help of ranges.
This notebook will create the table and fill it with the initial data. A second notebook will be used for increments of new and changed data.

This is an example of the expected output (salesRepSK is different
```
+----------+-------------+-------------+-----------+-------------------+-------------------+--------------------+-------+
|salesRepID|         name|       office| salesRepSK|          scd_start|            scd_end|                 md5|current|
+----------+-------------+-------------+-----------+-------------------+-------------------+--------------------+-------+
|a46add1...|      Z. Jane|     New York|          0|1990-01-01 00:00:00|2100-12-12 00:00:00|303db545462092a92...|   true|
|s1fedf1...|   P. Chapman|       Berlin|          1|1990-01-01 00:00:00|2100-12-12 00:00:00|14b094c31bf9e4149...|   true|
|d5e6f77...|     T. Crane|     New York|          2|1990-01-01 00:00:00|2100-12-12 00:00:00|6c062f95defda9dc3...|   true|
```




## Reading the source table

In [8]:
#EXTRACT
cc.set_connectionProfile("tutorial_op")

df_operational_sales_rep = spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "salesrep") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "salesRepID") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()

## Transforming the source to the dimension format
Select the data from the source table and transform it to the dimension format. Add the scd_start, scd_end and current columns.
The md5 hash is used to identify changes in the source table, and should be based on all fields that cause a change in the dimension.
The surrogate key will be automatically generated when writing to the table.

In [9]:
#TRANSFORM
df_operational_sales_rep.createOrReplaceTempView("dimSalesRepTemp")
df_dim_sales_rep = spark.sql("""
SELECT
  salesRepID,
  name,
  office,
  to_timestamp('1999-01-01','yyyy-MM-dd') as scd_start,
  to_timestamp('2100-12-12','yyyy-MM-dd') as scd_end,
  md5(concat(name, office)) as md5,
  True as current
FROM dimSalesRepTemp
""")

## Creating the dimension table
It is possible to create a delta table with an identity column. This column will be used as the surrogate key of the dimension. The metastore will create the table. Writing to the table will automatically generate the surrogate key. The format has to correspond with the dataframe that will be written to the table.

In [10]:
from delta.tables import DeltaTable, IdentityGenerator
from pyspark.sql.types import LongType, StringType, TimestampType, IntegerType, BooleanType

#Initialize the table with the identity column

DeltaTable.create(spark) \
    .tableName("dimSalesRep") \
    .addColumn("salesRepSK", LongType(), nullable=False, generatedAlwaysAs=IdentityGenerator(0, 1)) \
    .addColumn("salesRepID", IntegerType(), nullable=False) \
    .addColumn("name", StringType()) \
    .addColumn("office", StringType()) \
    .addColumn("scd_start", TimestampType()) \
    .addColumn("scd_end", TimestampType()) \
    .addColumn("md5", StringType()) \
    .addColumn("current", BooleanType()) \
    .property("delta.feature.identityColumns", "supported") \
    .execute()

## Populating the dimension table
Load the dataframe in the table. The identity column will be automatically generated. The data will be stored in spark-warehouse/dimsalesrep

In [11]:
#LOAD
#Write to the table (identity column will be auto-generated)
df_dim_sales_rep.write.format("delta").mode("overwrite").saveAsTable("dimSalesRep")

## Delete the spark session

In [12]:
spark.stop()
